# Data Processing Suite Tests

A few tests to verify whether the data processing suite functions as intended. Data are generated here and loaded into the sim GUI. Results from data processing operations are compared to the methods used below for validation. 

In [ ]:
# Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures

### Initial Dataset Creation

In [ ]:
# Some arbitrary unix posix start time
t0 = 1_700_000_000

# Dataset 1: starts earlier, 1-second sampling
t1 = np.arange(t0, t0 + 600, 1)
df1 = pd.DataFrame({
    "posix_time": t1,
    "sinewave": 2.0 * np.sin(2 * np.pi * (t1 - t1[0]) / 60)
})

# Dataset 2: starts later but ends later, 2-second sampling (used to vallidate whether merging cuts of tails)
t2 = np.arange(t0 + 30, t0 + 660, 2)
df2 = pd.DataFrame({
    "posix_time": t2,
    "sinewave": 5.0 * np.sin(2 * np.pi * (t2 - t2[0]) / 120)
})

df1.to_csv("data/sinewave_1sec.csv", index=False)
df2.to_csv("data/sinewave_2sec.csv", index=False)

In [ ]:
df1.info()

In [ ]:
df1.describe()

In [ ]:
df2.info()

In [ ]:
df2.describe()

In [ ]:
plt.plot(df1["posix_time"], df1["sinewave"], label="df1")
plt.plot(df2["posix_time"], df2["sinewave"], label="df2")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

### Load data from GUI output

The following operations were performed on the two datasets created above:
 - Derivative of sinewave
 - Rolling extremal difference (REXD) of sinewave (window sizes 25 and 50)
 - Rolling anchored-range difference (RAD) of sinewave (window sizes 25 and 50)
 - Rolling standard deviation (RSTDEV) of sinewave (window sizes 25 and 50)
 - Rolling variance (RVAR) of sinewave (window sizes 25 and 50)
 - Polynomial expansion of sinewave (2nd and 3rd order)
 - Remove single var (sin2)

For multiple vars:
 - Polynomial expansion of both sinewaves (2nd and 3rd order)
 - Create interaction term between both vars
 - Remove original two vars

An additional dataset is also created to test miscellaneous functionalities. The dataset contains 11 observations with integer values from 1 to 10. 10 will be included twice:
 - Remove duplicate observations (10)
 - Interpolate linearly
 - Forward fill
 - Back fill

In yet another file, structured similarly, but with 9th observation replaced with NaN:
 - Remove NaN

Datasets are merged on import, so all observations with NaNs are automatically removed after merging.

In [ ]:
df = pd.read_csv("data/gui_export_downsample.csv")
df.info()

In [ ]:
plt.plot(df1["posix_time"], df1["sinewave"], label="df1")
plt.scatter(df["posix_time"], df["sinewave"], alpha=0.4, color="orange", label="downsample")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
df = pd.read_csv("data/gui_export_derivatives.csv")
df.info()

In [ ]:
df.describe()

In [ ]:
# Plot both sinewaves after import
plt.plot(df1["posix_time"], df1["sinewave"], color="red", label="df1")
plt.plot(df2["posix_time"], df2["sinewave"], color="purple", label="df2")

plt.scatter(df["posix_time"], df["sinewave_1sec__sinewave"], alpha=0.2, label="GUI df1")
plt.scatter(df["posix_time"], df["sinewave_2sec__sinewave"], alpha=0.2, label="GUI df2")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

Sinewaves from both sets appear to match after being loaded and merged into the GUI. Cutoff times are consistent with expected results (i.e., both datasets begin at posix time of last set to begin and vice versa)

### Check 1: 1st and 2nd Derviatives

In [ ]:
df1['sinewave_derivative'] = df1['sinewave'].diff()
df1['sinewave_derivative_derivative'] = df1['sinewave_derivative'].diff()

# Sinewave vs derivative
plt.plot(df1["posix_time"], df1["sinewave"], label="df1")
plt.plot(df1["posix_time"], df1["sinewave_derivative"], label="df1 der")
plt.plot(df1["posix_time"], df1["sinewave_derivative_derivative"], label="df1 2nd der")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
# Sinewave vs derivatives
plt.plot(df1["posix_time"], df1["sinewave"], label="original df1")
plt.plot(df1["posix_time"], df1["sinewave_derivative"], label="original df1 der")
plt.plot(df1["posix_time"], df1["sinewave_derivative_derivative"], label="original df1 2nd der")

plt.scatter(df["posix_time"], df["sinewave_1sec__sinewave"], alpha=0.2, label="df1")
plt.scatter(df["posix_time"], df["sinewave_1sec__sinewave_derivative"], alpha=0.2, label="df1 der")
plt.scatter(df["posix_time"], df["sinewave_1sec__sinewave_derivative_derivative"], alpha=0.2, label="df1 2nd der")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

### Check 2: Rolling features

Check functionality of fixed-window rolling feature quantification methods:
 - Rolling extremal difference: difference between windowed min and max
 - Rolling anchored-range difference: maximal value between any point in the window and the value at index 0
 - Rolling standard deviation: self-explanatory
 - Rolling variation: self-explanatory

Verifying at windows of size 25 and 50.

In [ ]:
x = df1["sinewave"]
w = 25

df1["rexd_25"] = x.rolling(w, min_periods=1).max() - x.rolling(w, min_periods=1).min()
df1["rad_25"] = x.rolling(w, min_periods=1).apply(
    lambda a: abs(a - a[0]).max(),
    raw=True
)
df1["rstdev_25"] = x.rolling(w, min_periods=1).std()
df1["rvar_25"] = x.rolling(w, min_periods=1).var()

# Sinewave vs rolling features
plt.plot(df1["posix_time"], df1["sinewave"], label="df1")
plt.plot(df1["posix_time"], df1["rexd_25"], label="df1 rexd 25")
plt.plot(df1["posix_time"], df1["rad_25"], label="df1 rad 25")
plt.plot(df1["posix_time"], df1["rstdev_25"], label="df1 rstdev 25")
plt.plot(df1["posix_time"], df1["rvar_25"], label="df1 rvar 25")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
x = df1["sinewave"]
w = 50

df1["rexd_50"] = x.rolling(w, min_periods=1).max() - x.rolling(w, min_periods=1).min()
df1["rad_50"] = x.rolling(w, min_periods=1).apply(
    lambda a: abs(a - a[0]).max(),
    raw=True
)
df1["rstdev_50"] = x.rolling(w, min_periods=1).std()
df1["rvar_50"] = x.rolling(w, min_periods=1).var()

# Sinewave vs derivative
plt.plot(df1["posix_time"], df1["sinewave"], label="df1")
plt.plot(df1["posix_time"], df1["rexd_50"], label="df1 rexd 50")
plt.plot(df1["posix_time"], df1["rad_50"], label="df1 rad 50")
plt.plot(df1["posix_time"], df1["rstdev_50"], label="df1 rstdev 50")
plt.plot(df1["posix_time"], df1["rvar_50"], label="df1 rvar 50")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
df_roll = pd.read_csv("data/gui_export_rolling.csv")
df_roll.info()

In [ ]:
df_roll.describe()

In [ ]:
# Sinewave vs rolling features
plt.plot(df1["posix_time"], df1["sinewave"], label="df1")
plt.plot(df1["posix_time"], df1["rexd_25"], label="df1 rexd 25")
plt.plot(df1["posix_time"], df1["rad_25"], label="df1 rad 25")
plt.plot(df1["posix_time"], df1["rstdev_25"], label="df1 rstdev 25")
plt.plot(df1["posix_time"], df1["rvar_25"], label="df1 rvar 25")

plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave"], alpha=0.2, label="df1")
plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rexd_25"], alpha=0.2, label="df1 rexd 25")
plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rad_25"], alpha=0.2, label="df1 rad 25")
plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rstdev_25"], alpha=0.2, label="df1 rstdev 25")
plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rvar_25"], alpha=0.2, label="df1 rvar 25")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
# Sinewave vs rolling features
plt.plot(df1["posix_time"], df1["sinewave"], label="df1")
#plt.plot(df1["posix_time"], df1["rexd_25"], label="df1 rexd 25")
#plt.plot(df1["posix_time"], df1["rad_25"], label="df1 rad 25")
#plt.plot(df1["posix_time"], df1["rstdev_25"], label="df1 rstdev 25")
#plt.plot(df1["posix_time"], df1["rvar_25"], label="df1 rvar 25")

plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave"], alpha=0.2, label="df1")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rexd_25"], alpha=0.2, label="df1 rexd 25")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rad_25"], alpha=0.2, label="df1 rad 25")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rstdev_25"], alpha=0.2, label="df1 rstdev 25")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rvar_25"], alpha=0.2, label="df1 rvar 25")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
# Sinewave vs rolling features
#plt.plot(df1["posix_time"], df1["sinewave"], label="df1")
plt.plot(df1["posix_time"], df1["rexd_25"], label="df1 rexd 25")
#plt.plot(df1["posix_time"], df1["rad_25"], label="df1 rad 25")
#plt.plot(df1["posix_time"], df1["rstdev_25"], label="df1 rstdev 25")
#plt.plot(df1["posix_time"], df1["rvar_25"], label="df1 rvar 25")

#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave"], alpha=0.2, label="df1")
plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rexd_25"], alpha=0.2, label="df1 rexd 25")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rad_25"], alpha=0.2, label="df1 rad 25")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rstdev_25"], alpha=0.2, label="df1 rstdev 25")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rvar_25"], alpha=0.2, label="df1 rvar 25")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
# Sinewave vs rolling features
#plt.plot(df1["posix_time"], df1["sinewave"], label="df1")
#plt.plot(df1["posix_time"], df1["rexd_25"], label="df1 rexd 25")
plt.plot(df1["posix_time"], df1["rad_25"], label="df1 rad 25")
#plt.plot(df1["posix_time"], df1["rstdev_25"], label="df1 rstdev 25")
#plt.plot(df1["posix_time"], df1["rvar_25"], label="df1 rvar 25")

#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave"], alpha=0.2, label="df1")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rexd_25"], alpha=0.2, label="df1 rexd 25")
plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rad_25"], alpha=0.2, label="df1 rad 25")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rstdev_25"], alpha=0.2, label="df1 rstdev 25")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rvar_25"], alpha=0.2, label="df1 rvar 25")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
# Sinewave vs rolling features
#plt.plot(df1["posix_time"], df1["sinewave"], label="df1")
#plt.plot(df1["posix_time"], df1["rexd_25"], label="df1 rexd 25")
#plt.plot(df1["posix_time"], df1["rad_25"], label="df1 rad 25")
plt.plot(df1["posix_time"], df1["rstdev_25"], label="df1 rstdev 25")
#plt.plot(df1["posix_time"], df1["rvar_25"], label="df1 rvar 25")

#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave"], alpha=0.2, label="df1")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rexd_25"], alpha=0.2, label="df1 rexd 25")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rad_25"], alpha=0.2, label="df1 rad 25")
plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rstdev_25"], alpha=0.2, label="df1 rstdev 25")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rvar_25"], alpha=0.2, label="df1 rvar 25")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
# Sinewave vs rolling features
#plt.plot(df1["posix_time"], df1["sinewave"], label="df1")
#plt.plot(df1["posix_time"], df1["rexd_25"], label="df1 rexd 25")
#plt.plot(df1["posix_time"], df1["rad_25"], label="df1 rad 25")
#plt.plot(df1["posix_time"], df1["rstdev_25"], label="df1 rstdev 25")
plt.plot(df1["posix_time"], df1["rvar_25"], label="df1 rvar 25")

#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave"], alpha=0.2, label="df1")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rexd_25"], alpha=0.2, label="df1 rexd 25")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rad_25"], alpha=0.2, label="df1 rad 25")
#plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rstdev_25"], alpha=0.2, label="df1 rstdev 25")
plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rvar_25"], alpha=0.2, label="df1 rvar 25")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
# Sinewave vs rolling features
plt.plot(df1["posix_time"], df1["sinewave"], label="df1")
plt.plot(df1["posix_time"], df1["rexd_50"], label="df1 rexd 50")
plt.plot(df1["posix_time"], df1["rad_50"], label="df1 rad 50")
plt.plot(df1["posix_time"], df1["rstdev_50"], label="df1 rstdev 50")
plt.plot(df1["posix_time"], df1["rvar_50"], label="df1 rvar 50")

plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave"], alpha=0.2, label="df1")
plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rexd_50"], alpha=0.2, label="df1 rexd 50")
plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rad_50"], alpha=0.2, label="df1 rad 50")
plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rstdev_50"], alpha=0.2, label="df1 rstdev 50")
plt.scatter(df_roll["posix_time"], df_roll["sinewave_1sec__sinewave_rvar_50"], alpha=0.2, label="df1 rvar 50")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

Despite the GUI recreations exhibiting a 25-observation lag , this is consistent with what we expect since the dataset is truncated upon merge within the GUI.

### Check 3: Polynomial expansion

Check 2nd and 3rd order polynomial feature expansion

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(df1[["sinewave"]])

df_poly = pd.DataFrame(
    X_poly,
    columns=poly.get_feature_names_out(["sinewave"]),
    index=df1.index
)

df1 = pd.concat([df1, df_poly.drop(columns=["sinewave"])], axis=1)

# 3 deg
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(df1[["sinewave"]])

df_poly = pd.DataFrame(
    X_poly,
    columns=poly.get_feature_names_out(["sinewave"]),
    index=df1.index
)

df1 = pd.concat([df1, df_poly.drop(columns=["sinewave"])], axis=1)

df1.info()

In [ ]:
# Sinewave vs polynomial expansion
plt.plot(df1["posix_time"], df1["sinewave"], label="df1")
plt.plot(df1["posix_time"], df1["sinewave^2"], label="df1 poly 2")
plt.plot(df1["posix_time"], df1["sinewave^3"], label="df1 poly 3")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
df_polyexp = pd.read_csv("data/gui_export_polyexp.csv")
df_polyexp.info()

In [ ]:
# Sinewave vs rolling features
plt.plot(df1["posix_time"], df1["sinewave"], label="orig df1")
plt.plot(df1["posix_time"], df1["sinewave^2"], label="orig df1 poly 2")
plt.plot(df1["posix_time"], df1["sinewave^3"], label="orig df1 poly 3")

plt.scatter(df_polyexp["posix_time"], df_polyexp["sinewave_1sec__sinewave"], alpha=0.2, label="df1")
plt.scatter(df_polyexp["posix_time"], df_polyexp["sinewave_1sec__sinewave^2"], alpha=0.2, label="df1 poly 2")
plt.scatter(df_polyexp["posix_time"], df_polyexp["poly_sinewave_1sec__sinewave^2_3"], alpha=0.2, label="df1 poly 2")
plt.scatter(df_polyexp["posix_time"], df_polyexp["sinewave_1sec__sinewave^3"], alpha=0.2, label="df1 poly 3")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

GUI-produced data is consistent with processing pipeline output. Note that creating a polynomial expansion of a single feature will also produce all lower-order expansions as well (e.g., degree 3 polynomial produces x^3 *and* x^2.

In [ ]:
df1.info()

In [ ]:
df2.info()

In [ ]:
df1 = df1[["posix_time", "sinewave"]]

start = max(df1["posix_time"].min(), df2["posix_time"].min())
end = min(df1["posix_time"].max(), df2["posix_time"].max())

a = (df1[(df1["posix_time"] >= start) & (df1["posix_time"] <= end)].rename(columns={"sinewave": "sinewave_df1"}).sort_values("posix_time"))

b = (df2[(df2["posix_time"] >= start) & (df2["posix_time"] <= end)].rename(columns={"sinewave": "sinewave_df2"}).sort_values("posix_time"))

df_merged = pd.merge_asof(a, b, on="posix_time", direction="nearest").dropna().reset_index(drop=True)

df_merged.info()

In [ ]:
df_merged["interact"] = df_merged["sinewave_df1"] * df_merged["sinewave_df2"]
df_merged["averaged"] = (df_merged["sinewave_df1"] + df_merged["sinewave_df2"]) / 2
df_merged.info()

In [ ]:
# Polynomially expand df1 and df2 sinewaves
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(df_merged[["sinewave_df1", "sinewave_df2"]])

df_poly = pd.DataFrame(
    X_poly,
    columns=poly.get_feature_names_out(["sinewave_df1", "sinewave_df2"]),
    index=df_merged.index
)

df_merged = pd.concat([df_merged, df_poly.drop(columns=["sinewave_df1", "sinewave_df2"])], axis=1)

# 3 deg
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(df_merged[["sinewave_df1", "sinewave_df2"]])

df_poly = pd.DataFrame(
    X_poly,
    columns=poly.get_feature_names_out(["sinewave_df1", "sinewave_df2"]),
    index=df_merged.index
)

df_merged = pd.concat([df_merged, df_poly.drop(columns=["sinewave_df1", "sinewave_df2"])], axis=1)

df_merged = df_merged.loc[:, ~df_merged.columns.duplicated()]

df_merged.info()

In [ ]:
# Sinewave vs polynomial expansion
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1"], label="df1")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df2"], label="df2")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1^2"], label="df1 poly 2")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1 sinewave_df2"], label="df1 df2")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df2^2"], label="df2 poly 2")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1"], label="df1")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df2"], label="df2")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1^3"], label="df1 poly 3")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1^2 sinewave_df2"], label="df1 p2 df2")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1 sinewave_df2^2"], label="df1 df2 p2")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df2^3"], label="df2 poly 3")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
df_poly_inter_avg = pd.read_csv("data/gui_export_polyexp_interact_avging.csv")
df_poly_inter_avg = df_poly_inter_avg.drop(columns=[c for c in df_poly_inter_avg.columns if c.startswith("poly_") and c.endswith("_3")], errors="ignore")
df_poly_inter_avg.info()

In [ ]:
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1"], label="df1")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df2"], label="df2")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1^2"], label="df1 poly 2")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1 sinewave_df2"], label="df1 df2")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df2^2"], label="df2 poly 2")

plt.scatter(df_poly_inter_avg["posix_time"], df_poly_inter_avg["sinewave_1sec__sinewave^2"], alpha=0.2, label="df1 poly 2")
plt.scatter(df_poly_inter_avg["posix_time"], df_poly_inter_avg["sinewave_1sec__sinewave * sinewave_2sec__sinewave"], alpha=0.2, label="df1 df2")
plt.scatter(df_poly_inter_avg["posix_time"], df_poly_inter_avg["sinewave_2sec__sinewave^2"], alpha=0.2, label="df2 poly 2")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1"], label="df1")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df2"], label="df2")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1^3"], label="df1 poly 3")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1^2 sinewave_df2"], label="df1 p2 df2")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1 sinewave_df2^2"], label="df1 df2 p2")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df2^3"], label="df2 poly 3")

plt.scatter(df_poly_inter_avg["posix_time"], df_poly_inter_avg["sinewave_1sec__sinewave^3"], alpha=0.2, label="df1 poly 3")
plt.scatter(df_poly_inter_avg["posix_time"], df_poly_inter_avg["sinewave_1sec__sinewave^2 * sinewave_2sec__sinewave"], alpha=0.2, label="df1 p2 df2")
plt.scatter(df_poly_inter_avg["posix_time"], df_poly_inter_avg["sinewave_1sec__sinewave * sinewave_2sec__sinewave^2"], alpha=0.2, label="df1 df2 p2")
plt.scatter(df_poly_inter_avg["posix_time"], df_poly_inter_avg["sinewave_2sec__sinewave^3"], alpha=0.2, label="df2 poly 3")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

All polynomially expanded features match.

In [ ]:
# Create interaction terms for df1 and df2 sinewave
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1"], label="df1")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df2"], label="df2")
plt.plot(df_merged["posix_time"], df_merged["interact"], label="df1 df2")

plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1"], label="df1")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df2"], label="df2")
plt.plot(df_merged["posix_time"], df_merged["interact"], label="df1 df2")

plt.scatter(df_poly_inter_avg["posix_time"], df_poly_inter_avg["additive"], alpha=0.2, label="df1 df2")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()



In [ ]:
# Create average of both df1 and df2 sinewaves
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1"], label="df1")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df2"], label="df2")
plt.plot(df_merged["posix_time"], df_merged["averaged"], label="avg")

plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
plt.plot(df_merged["posix_time"], df_merged["sinewave_df1"], label="df1")
plt.plot(df_merged["posix_time"], df_merged["sinewave_df2"], label="df2")
plt.plot(df_merged["posix_time"], df_merged["averaged"], label="orig avg")

plt.scatter(df_poly_inter_avg["posix_time"], df_poly_inter_avg["averaged"], alpha=0.2, label="avg")
plt.xlabel("POSIX time")
plt.ylabel("Sinewave")
plt.legend()
plt.show()

In [ ]:
df3 = pd.DataFrame({
    "test": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10]
})
#df4.loc[df3["test"] == 8, "test"] = np.nan

df3.to_csv("data/test1.csv", index=False)
print(df3.to_string(index=False))

# Remove duplicate observations (10)
#Remove single observation (2)
#Remove multiple observations (3-9)
#Interpolate linearly

#Remove observation (9) and forward fill

#Remove NaN

In [ ]:
df4 = df3.drop_duplicates(subset=["test"])
print(df4.to_string(index=False))

df4_gui = pd.read_csv("data/gui_export_rem_dups.csv")
print(df4_gui.to_string(index=False))

In [ ]:
df5 = pd.DataFrame({
    "posix_time": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    "test": [1, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 10]
})

df5.to_csv("data/test2.csv", index=False)
print(df5.to_string(index=False))

In [ ]:
df6 = df5.copy()
df6["test"] = df6["test"].ffill()
print(df6.to_string(index=False))

df6_gui = pd.read_csv("data/gui_export_ffill.csv")
print(df6_gui.to_string(index=False))

In [ ]:
df7 = df5.copy()
df7["test"] = df7["test"].bfill()
print(df7.to_string(index=False))

df7_gui = pd.read_csv("data/gui_export_bfill.csv")
print(df7_gui.to_string(index=False))

In [ ]:
df8 = df5.copy()
df8["test"] = df8["test"].interpolate(method="linear")
print(df8.to_string(index=False))

df8_gui = pd.read_csv("data/gui_export_1inter.csv")
print(df8_gui.to_string(index=False))

In [ ]:
# Check remove nans
df9 = df5.dropna()
print(df9.to_string(index=False))

df9_gui = pd.read_csv("data/gui_export_rem_nans.csv")
print(df9_gui.to_string(index=False))

In [ ]:
# Check remove multiple vars (create 1st and 2nd derivatives of test and delete test and posix time from df)
df10_gui = pd.read_csv("data/gui_export_rem_vars.csv")
print(df10_gui.columns)

In [ ]:
# Check remove single var (create 1st derivative of test and just delete test from df)
df11_gui = pd.read_csv("data/gui_export_rem_var.csv")
print(df11_gui.columns)